# Exercise 7 : Combining Data

Compute the 2024 population of each World Bank income group and its share aged 65 and over. Count
each economy once, and keep every row that you leave out visible in an audit.

Run the first cell : it sets `DATA` to the course's `datasets` folder and `SOURCES` to the two
indicator files with their labels. Read `datasets/wb-population-README.md` first. The files are
unchanged World Bank downloads, retrieved on 10 October 2026 :

- `wb-wdi-population-total.csv` and `wb-wdi-population-65plus.csv` : World Development Indicators,
  with one row per economy or aggregate and one column per year
- `wb-class-economies.csv` : the income classification, one row per economy or aggregate
- `wb-class-groups.csv` : the group memberships, one row per economy and group

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets'
SOURCES = {'total': 'wb-wdi-population-total.csv', '65plus': 'wb-wdi-population-65plus.csv'}

## Step 1 : Appending

Read the two indicator files into the list `frames`. Open one of them in a text editor first : four
metadata lines precede the header, so use `skiprows=4`. Every line ends with a comma, which creates
an empty column `Unnamed: 70`. Check that it is empty and drop it.

Add the labels of `SOURCES` as a `source` column and concatenate the frames once into
`indicators`, with a fresh integer index. Check that :

- `value_counts()` on `source` gives 266 rows per file, 532 in total
- No pair of `Country Code` and `Indicator Code` occurs twice
- Every country code is three capital letters : `str.fullmatch(r'[A-Z]{3}')`

Count the missing 2024 values as `missing_2024`. Which row is empty? Why is an empty value
different from zero people?

In [ ]:
# Your code here


## Step 2 : Duplicate keys

Keep the `Country Code`, `Indicator Code` and `2024` columns of `indicators` as `population_2024`.
Read `wb-class-groups.csv` into `groups`. Select every row whose `WB_Country_Code` occurs more than
once, keeping all occurrences, as `duplicate_groups`, and print the memberships of the Netherlands.

Try a left merge of `population_2024` with `groups` on `Country Code` and `WB_Country_Code`, with
`validate='many_to_one'`. Catch only `pd.errors.MergeError` and print its message. Why does a
membership list repeat every economy? Which of the two tables must have unique keys for this merge?

In [ ]:
# Your code here


## Step 3 : Validated merge

Read `wb-class-economies.csv` into `economies`. Find and drop its one completely empty row with
`isna().all(axis=1)`, and check that `Code` is complete and unique.

Merge `population_2024` with `economies` into `combined`, with `how='left'`,
`validate='many_to_one'` and `indicator=True`. Check that `combined` has as many rows as
`population_2024` and the same 2024 total. Each country appears twice on the left, once per
indicator : why does the validation accept that?

In [ ]:
# Your code here


## Step 4 : Audit

Split `combined` into three parts and check that together they hold all its rows :

- `unmatched` : the `left_only` rows, whose code is not in `economies`
- `aggregates` : the matched rows without an income group, such as regions and the world
- `economies_2024` : the rows with an income group

Repeat the merge as `outer_audit`, with `how='outer'` and the same validation. Select its
`right_only` rows as `unused_economies` : economies without a row in the indicator files.

Compute three totals for 2024 :

- `world` : the total population of `WLD`
- `economy_total` : the total population summed over `economies_2024`
- `gap` : the difference between the two

Which economy explains the gap? Compare the High income aggregate, `HIC`, with the sum of its member
economies, then look at `unused_economies`. Explain in the answer cell whether leaving out the
aggregates and the unmatched rows loses population.

In [ ]:
# Your code here


In [ ]:
# Your code here


Your answer here.

## Step 5 : Income groups

Put the two indicators side by side per income group with `pivot_table(..., aggfunc='sum')` on
`economies_2024`, as `by_income`, and add the column `share_65plus`. Why is a sum safe here, while
the same call on a table that repeats an economy would hide a defect? The slide *Aggregating
Duplicate Pairs* shows the risk.

In [ ]:
# Your code here


## If time permits :

1. Melt the year columns of the total-population file into `long`, keeping `Country Code` and
   `Country Name`, with the columns `year` and `population`. Predict the number of rows, then
   check it. Look up the Netherlands in 2024 by its labels : 17,993,485. Pivot `long` back into
   `restored` and look up the Netherlands in 2024 and in 1960 : 11,486,631.
2. The notebook builds `bad`, an unchecked left join of the 2024 population of the economies onto
   `groups`. Compute `original_total`, `bad_total` and `extra`. Count the rows per economy with
   `value_counts()` on `Country Code` as `repeats`, and find the smallest and largest count.
   Explain in two or three sentences in the answer cell which table repeats its keys, how often,
   why the total grows and what repairs it. Does `validate='many_to_many'` show that the result is
   correct?

In [ ]:
# Your code here


In [ ]:
classified = economies.loc[economies['Income group'].notna(), 'Code']
totals_wide = frames[0]
population = totals_wide.loc[totals_wide['Country Code'].isin(classified),
                             ['Country Code', '2024']]
bad = population.merge(groups, left_on='Country Code', right_on='WB_Country_Code', how='left')

In [ ]:
# Your code here


Your answer here.